# Meta-Learning for Adaptive Trading

This notebook demonstrates how to use meta-learning to create adaptive trading strategies that quickly adjust to changing market conditions.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from src.models.meta_learner import MAMLModel
from src.data.binance_fetcher import BinanceFetcher
from src.features.feature_engineering import FeatureEngineer

## Configuration

Load and configure meta-learning settings.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Meta-learning parameters
META_PARAMS = {
    'inner_lr': 0.01,
    'meta_lr': 0.001,
    'num_inner_steps': 5,
    'task_batch_size': 32
}

## Data Preparation

Fetch historical data and prepare tasks for meta-learning.

In [ ]:
# Initialize data fetcher
fetcher = BinanceFetcher(config['data']['binance'])

# Fetch historical data
data = await fetcher.fetch_historical_data()

# Initialize feature engineer
engineer = FeatureEngineer(
    input_file='data/raw/BTCUSDT_data.parquet',
    output_dir='data/processed',
    config=config
)

# Generate features
await engineer.generate_features()

## Task Generation

Create tasks for meta-learning based on different market regimes.

In [ ]:
class TaskGenerator:
    def __init__(self, data, window_size=100, stride=20):
        self.data = data
        self.window_size = window_size
        self.stride = stride
        
        # Calculate volatility for regime identification
        self.returns = data['close'].pct_change()
        self.volatility = self.returns.rolling(window=20).std()
        
        # Identify market regimes
        self.regimes = {
            'low_vol': self.volatility < self.volatility.quantile(0.33),
            'med_vol': (self.volatility >= self.volatility.quantile(0.33)) & 
                      (self.volatility < self.volatility.quantile(0.66)),
            'high_vol': self.volatility >= self.volatility.quantile(0.66)
        }
    
    def create_task(self, start_idx):
        """Create a single task from data starting at start_idx."""
        window = self.data.iloc[start_idx:start_idx + self.window_size]
        
        # Split into support and query sets
        split_idx = int(0.8 * len(window))
        support_data = window.iloc[:split_idx]
        query_data = window.iloc[split_idx:]
        
        return support_data, query_data
    
    def sample_tasks(self, num_tasks, regime=None):
        """Sample tasks from specified regime or all data."""
        if regime:
            valid_indices = np.where(self.regimes[regime])[0]
        else:
            valid_indices = np.arange(len(self.data) - self.window_size)
        
        # Sample start indices
        start_indices = np.random.choice(
            valid_indices,
            size=num_tasks,
            replace=False
        )
        
        return [self.create_task(idx) for idx in start_indices]

# Initialize task generator
task_generator = TaskGenerator(data)

# Sample tasks from different regimes
low_vol_tasks = task_generator.sample_tasks(10, regime='low_vol')
high_vol_tasks = task_generator.sample_tasks(10, regime='high_vol')

## Meta-Learning Training

Train the meta-learning model across different market regimes.

In [ ]:
# Initialize meta-learning model
model = MAMLModel(config, **META_PARAMS)

# Train meta-learner
history = model.meta_learn(
    task_generator=task_generator.sample_tasks,
    num_tasks=100,
    num_epochs=50
)

# Plot training progress
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['meta_loss'])
plt.title('Meta-Learning Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(history['adaptation_metrics'])
plt.title('Adaptation Performance')
plt.xlabel('Epoch')
plt.ylabel('Performance')

plt.tight_layout()
plt.show()

## Market Regime Analysis

Analyze model performance across different market regimes.

In [ ]:
def evaluate_regime(model, tasks):
    """Evaluate model performance on a set of tasks."""
    performances = []
    
    for support_data, query_data in tasks:
        # Adapt model to support data
        adapted_model = model.adapt_to_market(support_data)
        
        # Evaluate on query data
        with torch.no_grad():
            _, metrics = adapted_model(query_data)
        
        performances.append(metrics)
    
    return pd.DataFrame(performances)

# Evaluate performance in different regimes
low_vol_perf = evaluate_regime(model, low_vol_tasks)
high_vol_perf = evaluate_regime(model, high_vol_tasks)

# Plot performance comparison
plt.figure(figsize=(10, 5))
plt.boxplot([low_vol_perf['accuracy'], high_vol_perf['accuracy']],
            labels=['Low Volatility', 'High Volatility'])
plt.title('Model Performance Across Market Regimes')
plt.ylabel('Accuracy')
plt.show()

## Real-Time Adaptation

Demonstrate real-time model adaptation to current market conditions.

In [ ]:
def simulate_trading(model, data, window_size=100):
    """Simulate trading with real-time adaptation."""
    results = []
    
    for i in range(window_size, len(data), 20):
        # Get recent data for adaptation
        recent_data = data.iloc[i-window_size:i]
        
        # Adapt model
        adapted_model = model.adapt_to_market(recent_data)
        
        # Make predictions
        future_data = data.iloc[i:i+20]
        with torch.no_grad():
            predictions, _ = adapted_model(future_data)
        
        results.append({
            'timestamp': future_data.index[-1],
            'predictions': predictions.numpy(),
            'actual': future_data['close'].values,
            'volatility': task_generator.volatility.iloc[i]
        })
    
    return pd.DataFrame(results)

# Run trading simulation
simulation_results = simulate_trading(model, data)

# Plot results
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.plot(simulation_results['actual'], label='Actual')
plt.plot(simulation_results['predictions'], label='Predicted')
plt.title('Price Predictions')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(simulation_results['volatility'],
           np.abs(simulation_results['predictions'] - simulation_results['actual']))
plt.title('Prediction Error vs Volatility')
plt.xlabel('Volatility')
plt.ylabel('Absolute Error')

plt.tight_layout()
plt.show()

## Performance Analysis

Analyze the trading performance and adaptation effectiveness.

In [ ]:
def calculate_metrics(results):
    """Calculate trading performance metrics."""
    metrics = {
        'mse': np.mean((results['predictions'] - results['actual'])**2),
        'mae': np.mean(np.abs(results['predictions'] - results['actual'])),
        'correlation': np.corrcoef(results['predictions'], results['actual'])[0,1]
    }
    
    # Calculate by volatility regime
    vol_quantiles = np.percentile(results['volatility'], [33, 66])
    
    for regime, mask in [
        ('low_vol', results['volatility'] <= vol_quantiles[0]),
        ('med_vol', (results['volatility'] > vol_quantiles[0]) & 
                    (results['volatility'] <= vol_quantiles[1])),
        ('high_vol', results['volatility'] > vol_quantiles[1])
    ]:
        regime_data = results[mask]
        metrics[f'{regime}_mse'] = np.mean(
            (regime_data['predictions'] - regime_data['actual'])**2
        )
    
    return metrics

# Calculate and display metrics
metrics = calculate_metrics(simulation_results)
print("Performance Metrics:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

## Save Model

Save the trained meta-learning model for future use.

In [ ]:
# Save model
save_path = 'models/meta_learned_model.pt'
model.save_meta_learned(save_path)
print(f"Model saved to {save_path}")